# Investment Research Agent
**AAI-520 Final Project — University of San Diego**

**Team:** Viviana Garzon Silva · Obinna Edeh · Dina Othman  
**GitHub:** [vgsilva-eng/Natural-Language-Processing](https://github.com/vgsilva-eng/Natural-Language-Processing)

## Project Goal

This project builds a multi-agent system that researches a stock ticker using live market, financial, and news data. Three specialist agents contribute focused analysis, while the ResearchAgent coordinates their results into an investment report. The notebook also demonstrates memory, reflection, and three workflow patterns.

| Component | Role |
|---|---|
| **MarketAgent** | Computes price, return, moving-average, and volume signals from Yahoo Finance data. |
| **EarningsAgent** | Retrieves financial results and derives profitability and valuation signals. |
| **NewsAgent** | Classifies recent articles and summarizes investor-relevant themes. |
| **ResearchAgent** | Plans the task, runs the specialists, synthesizes their output, saves memory, and reflects on report quality. |
| **Prompt chaining** | Ingests, cleans, classifies, extracts, and summarizes news in ordered stages. |
| **Routing** | Classifies an input and dispatches it to the matching specialist. |
| **Evaluator-optimizer** | Scores a draft, supplies feedback, and rewrites it until the threshold or iteration limit is reached. |

The examples use **AAPL**, but the system accepts other valid ticker symbols.

## 0. Setup

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dotenv import load_dotenv
load_dotenv(repo_root / ".env")

TICKER = "AAPL"
print("Setup complete. Repo root:", repo_root)

## 1. Data Tools

The tools form the system's data layer. Yahoo Finance supplies price history and company financials, while NewsAPI supplies current articles. Running them independently first makes provider or credential problems easier to distinguish from agent logic.

In [ ]:
from tools.yfinance_tool import get_stock_data, get_financials

market = get_stock_data(TICKER)
print(f"Ticker:        {market['ticker']}")
print(f"Current price: ${market['current_price']:,.2f}")
print(f"Change today:  {market['change_pct']:+.2f}%")
print(f"52W range:     ${market['week_52_low']} – ${market['week_52_high']}")
print(f"MA50:          {market['ma_50']}")
print(f"History rows:  {len(market['history'])}")

In [ ]:
fin = get_financials(TICKER)
print(f"Sector:        {fin['sector']}")
print(f"Revenue:       {fin['revenue']}")
print(f"Net income:    {fin['net_income']}")
print(f"P/E ratio:     {fin['pe_ratio']}")

In [ ]:
from tools.news_tool import fetch_news

articles = fetch_news(TICKER, lookback_days=7, max_articles=5)
print(f"Articles found: {len(articles)}\n")
for a in articles:
    print(f"  [{a['source']}] {a['title']}")

## 2. Market Agent

MarketAgent converts raw price history into deterministic indicators, including period return, 20-day and 50-day moving averages, and volume comparison. It does not need an LLM, so its calculations remain reproducible from the same input data.

In [ ]:
from agents.market_agent import MarketAgent

market_agent = MarketAgent()
market_result = market_agent.analyze(TICKER, period="3mo")

print(f"Ticker:          {market_result['ticker']} as of {market_result['as_of']}")
print(f"Price:           ${market_result['current_price']:,.2f}")
print(f"Period return:   {market_result['period_return_pct']:+.2f}%")
print(f"Trend:           {market_result['trend']}")
print(f"Crossover:       {market_result['crossover']}")
print(f"Volume ratio:    {market_result['volume_ratio']}x 20-session avg")
print(f"\nSummary: {market_result['summary']}")

In [ ]:
from IPython.display import SVG, display
from tools.market_visualization import save_price_chart

chart_path = save_price_chart(market_result, repo_root / "market_price_chart.svg")
display(SVG(filename=str(chart_path)))

## 3. Earnings Agent

EarningsAgent turns financial statement fields into a compact fundamental snapshot. It reports revenue, net income, earnings per share, valuation, margin, and plain-language signals while safely handling values that a provider may omit.

In [ ]:
from agents.earnings_agent import EarningsAgent

def fmt_currency(v, d=0):
    return f"${v:,.{d}f}" if v is not None else "N/A"

def fmt_number(v, d=2):
    return f"{v:.{d}f}" if v is not None else "N/A"

def fmt_pct(v):
    return f"{v:.2f}%" if v is not None else "N/A"

earnings_agent = EarningsAgent()
earnings_result = earnings_agent.analyze(TICKER)

print(f"Ticker:             {earnings_result['ticker']}")
print(f"Sector:             {earnings_result['sector'] or 'N/A'}")
print(f"Industry:           {earnings_result['industry'] or 'N/A'}")
print(f"Revenue:            {fmt_currency(earnings_result['revenue'])}")
print(f"Net Income:         {fmt_currency(earnings_result['net_income'])}")
print(f"EPS:                {fmt_currency(earnings_result['eps'], 2)}")
print(f"P/E Ratio:          {fmt_number(earnings_result['pe_ratio'])}")
print(f"Forward P/E:        {fmt_number(earnings_result['forward_pe'])}")
print(f"Net Profit Margin:  {fmt_pct(earnings_result['net_margin_pct'])}")
print("\nFinancial Signals:")
for signal in earnings_result['signals']:
    print(f"  - {signal}")

## 4. News Agent

NewsAgent fetches recent articles, uses the language model to classify each article into a financial topic, and summarizes the most relevant themes. The summary is grounded in the fetched titles and descriptions.

In [ ]:
from agents.news_agent import NewsAgent

news_agent = NewsAgent()
news_articles = news_agent.fetch_and_classify(TICKER, lookback_days=7)

print(f"Articles classified: {len(news_articles)}\n")
for article in news_articles[:5]:
    print(f"  [{article['category']:10s}] {article['title']}")

In [ ]:
news_summary = news_agent.summarize(news_articles)
print("News Summary:\n")
print(news_summary)

In [ ]:
# Package the specialist results for comparison with the final orchestrated report.
specialist_results = {
    "market": market_result,
    "earnings": earnings_result,
    "news": {
        "articles_analyzed": len(news_articles),
        "summary": news_summary,
    },
}

print(f"Market data source:      {market_result['data_source']}")
print(f"Earnings data source:    {earnings_result['data_source']}")
print(f"News articles analyzed:  {len(news_articles)}")
print("Earnings signals:")
for signal in earnings_result['signals']:
    print(f"  - {signal}")

## 5. Workflow 1: Prompt Chaining

Prompt chaining breaks a larger language task into five observable stages: ingest, preprocess, classify, extract facts, and summarize. Each stage receives the prior stage's output. This structure makes intermediate results inspectable and reduces the ambiguity of a single large prompt.

In [ ]:
from workflows.prompt_chaining import run_prompt_chain

chain_result = run_prompt_chain(TICKER, lookback_days=7)

print(f"Articles ingested:    {chain_result['articles_ingested']}")
print(f"After deduplication:  {chain_result['articles_after_dedup']}")
print("\nKey facts:")
for fact in chain_result['key_facts']:
    print(f"  - {fact}")

In [ ]:
print("News Summary:\n")
print(chain_result['summary'])

## 6. Workflow 2: Routing and Dispatch

The router first chooses `earnings`, `news`, or `market` from the input text. The dispatch function then instantiates that specialist and executes its analysis. The first example shows classification across three inputs; the second proves that the selected route is actually executed.

In [ ]:
import os
import anthropic
from workflows.routing import classify_route, route_to_specialist

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

samples = [
    "Apple reported quarterly EPS of $1.64, beating estimates by $0.08. Revenue grew 5% YoY.",
    "Analyst upgrades AAPL to Buy, citing strong services momentum.",
    "AAPL is trading above its 50-day moving average on above-average volume.",
]

for content in samples:
    route = classify_route(content, client)
    print(f"Route: {route:10s} | {content[:80]}")

print()
print("Dispatch demonstration:")
dispatch_result = route_to_specialist(samples[2], ticker=TICKER)
print(f"Selected specialist: {dispatch_result['route']}")
print(f"Returned ticker:     {dispatch_result['result'].get('ticker')}")
print(f"Returned signals:    {len(dispatch_result['result'].get('signals', []))}")

## 7. Workflow 3: Evaluator-Optimizer

This workflow evaluates a draft against accuracy, balance, clarity, and specificity criteria. If the score is below the target, the optimizer rewrites the draft using the evaluator's feedback. The recorded history makes the iteration process and stopping decision visible.

In [ ]:
from workflows.evaluator_optimizer import run_evaluator_optimizer

eval_result = run_evaluator_optimizer(
    draft=chain_result['summary'],
    threshold=8,
    max_iterations=3
)

print(f"Iterations: {eval_result['iterations']}\n")
for h in eval_result['history']:
    print(f"  Iteration {h['iteration']}: score {h['score']}/10")
    print(f"  Feedback: {h['feedback'][:120]}...\n")

In [ ]:
print("Final refined draft:\n")
print(eval_result['final_draft'])

## 8. Research Agent: Integrated Pipeline

ResearchAgent is the orchestrator. It creates a research plan, runs MarketAgent, EarningsAgent, and NewsAgent, synthesizes their outputs, and saves a short result in persistent memory. Reflection then critiques the report for a further improvement cycle.

In [ ]:
from agents.research_agent import ResearchAgent

agent = ResearchAgent()
report = agent.run(TICKER)

print("Research Plan:")
for i, step in enumerate(report['steps'], 1):
    print(f"  {i}. {step}")

In [ ]:
print("Market Snapshot:")
snap = report['market_snapshot']
print(f"  Price:     ${snap['price']:,.2f} ({snap['change_pct']:+.2f}%)")
print(f"  52W range: ${snap['week_52_low']} – ${snap['week_52_high']}")

print("\nFinancials Snapshot:")
fin = report['financials_snapshot']
print(f"  Revenue:    {fin['revenue']}")
print(f"  Net income: {fin['net_income']}")
print(f"  P/E ratio:  {fin['pe_ratio']}")
print(f"  Sector:     {fin['sector']}")

In [ ]:
print("Investment Analysis:\n")
print(report['analysis'])

In [ ]:
reflection = agent.reflect(report)
print("Self-Critique:\n")
print(reflection)

### Persistent Memory Demonstration

The first ResearchAgent run saves the ticker's latest price and summary. A new agent instance then loads the same record from disk, demonstrating that memory persists beyond the original object and can provide context to a later run.

In [ ]:
saved_memory = agent.memory.load().get(TICKER, {})
new_agent_instance = ResearchAgent()
reloaded_memory = new_agent_instance.memory.load().get(TICKER, {})

print(f"Memory saved for {TICKER}:   {bool(saved_memory)}")
print(f"New instance reloaded it:    {saved_memory == reloaded_memory}")
print(f"Remembered last price:       {reloaded_memory.get('last_price')}")
print(f"Remembered summary preview:  {reloaded_memory.get('summary', '')[:160]}...")

---
## 9. Conclusion

The completed system demonstrates a working multi-agent investment research pipeline. Live runs combine market trends, company financials, and recent news; prompt chaining exposes intermediate reasoning stages; routing sends content to an appropriate specialist; and evaluator-optimizer records feedback-driven revision. The ResearchAgent coordinates all three specialists, persists a compact memory for later sessions, and produces a self-critique.

### Results

- The three specialist agents return structured results that can be inspected independently and synthesized together.
- Routing demonstrates both classification and actual dispatch to a specialist.
- Evaluator history shows the score and feedback produced during each iteration.
- Persistent memory can be reloaded by a new ResearchAgent instance.
- Automated tests cover agents, data handling, routing, summarization, visualization, prompt-chain fallbacks, and Anthropic response blocks.

### Limitations

- Results depend on third-party API availability, coverage, rate limits, and the accuracy of Yahoo Finance and NewsAPI data.
- News classification and generated analysis may vary between model calls and should be reviewed before making decisions.
- The report uses public data and simplified indicators; it does not include portfolio constraints, transaction costs, or personalized risk tolerance.
- This project is an educational research tool and does not provide financial advice.

### Next Steps

Future work could add source citations inside the final narrative, cache provider responses, compare multiple tickers, add historical backtesting, and evaluate generated reports against a larger labeled benchmark.